In [1]:
#!/usr/bin/env python3
import os
import argparse
import numpy as np
from PIL import Image
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import warnings

from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, IsolationForest, VotingClassifier
from sklearn.preprocessing import RobustScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_auc_score
)
from skimage.feature import local_binary_pattern, hog
from skimage.filters import sobel, gabor

warnings.filterwarnings("ignore")

# --------------------------
# Feature extraction
# --------------------------
def extract_features_from_path(path):
    try:
        img = Image.open(path).convert("L").resize((128, 128))
        arr = np.array(img).astype(np.float32)

        # LBP
        lbp = local_binary_pattern(arr, P=8, R=1, method="uniform")
        lbp_hist, _ = np.histogram(lbp.ravel(), bins=np.arange(0, 11), range=(0, 10))
        lbp_hist = lbp_hist.astype(float) / (lbp_hist.sum() + 1e-8)

        # HOG
        hog_feat = hog(arr, pixels_per_cell=(16, 16), cells_per_block=(2, 2), feature_vector=True)

        # Gabor (3 angles)
        gabor_features = []
        for theta in (0, np.pi/4, np.pi/2):
            real, imag = gabor(arr, frequency=0.6, theta=theta)
            gabor_features.append(np.mean(real))
            gabor_features.append(np.mean(imag))

        # Sobel
        sobel_mean = np.mean(sobel(arr))

        feat = np.hstack([lbp_hist, hog_feat, gabor_features, sobel_mean])
        return np.array(feat, dtype=np.float32)
    except Exception as e:
        print(f"[WARN] Could not extract features from {path}: {e}")
        return None


def load_dir_features(directory, label):
    X, y = [], []
    if not os.path.exists(directory):
        print(f"[WARN] Directory not found: {directory}")
        return np.array(X), np.array(y)

    files = [os.path.join(root, f)
             for root, _, fnames in os.walk(directory)
             for f in fnames if f.lower().endswith((".png", ".jpg", ".jpeg", ".bmp", ".tiff"))]

    print(f"[INFO] Found {len(files)} images in {directory}")
    for p in tqdm(files, desc=f"Extracting {os.path.basename(directory)}"):
        feat = extract_features_from_path(p)
        if feat is not None:
            X.append(feat)
            y.append(label)
    return np.array(X), np.array(y)


def clean_nan_inf_and_variance(X, y, var_eps=1e-12):
    mask_bad = np.any(np.isnan(X) | np.isinf(X), axis=1)
    if np.any(mask_bad):
        print(f"[CLEAN] Removing {mask_bad.sum()} samples with NaN/Inf")
        X = X[~mask_bad]
        y = y[~mask_bad]

    vars_ = np.var(X, axis=0)
    mask_keep_cols = vars_ > var_eps
    if not np.all(mask_keep_cols):
        print(f"[CLEAN] Removing {np.sum(~mask_keep_cols)} constant feature columns")
        X = X[:, mask_keep_cols]
    return X, y, mask_keep_cols


def ensure_dir(d):
    if not os.path.exists(d):
        os.makedirs(d, exist_ok=True)


# --------------------------
# Main
# --------------------------
def main(
    benign_dir="dataset/benign",
    malicious_dir="dataset/malicious",
    results_dir="results",
    saved_models_dir="saved_models",
    iso_percentile=95,
    ensemble_weight=0.6,
    iso_weight=0.4,
    test_size=0.2,
    random_state=42,
    epochs=3
):
    ensure_dir(results_dir)
    ensure_dir(saved_models_dir)

    # Load features
    print("[STEP] Loading features...")
    benign_X, benign_y = load_dir_features(benign_dir, 0)
    malicious_X, malicious_y = load_dir_features(malicious_dir, 1)

    if benign_X.size == 0 and malicious_X.size == 0:
        print("❌ No images found. Exiting.")
        return

    X = np.vstack([benign_X, malicious_X])
    y = np.hstack([benign_y, malicious_y])

    print(f"[INFO] Raw feature shape: {X.shape}, labels: {y.shape}")

    # Clean
    X, y, keep_cols_mask = clean_nan_inf_and_variance(X, y)
    print(f"[INFO] After cleaning: {X.shape}")

    scaler = RobustScaler()
    X_scaled = scaler.fit_transform(X)

    # Isolation Forest
    iso = IsolationForest(n_estimators=200, contamination=0.02, random_state=random_state)
    iso.fit(X_scaled[y == 0])

    iso_scores_all = -iso.decision_function(X_scaled)
    iso_scores_benign = iso_scores_all[y == 0]
    percentile_val = np.percentile(iso_scores_benign, iso_percentile)
    print(f"[INFO] IsolationForest {iso_percentile}th percentile = {percentile_val:.6f}")

    # Histogram
    plt.figure(figsize=(8, 4))
    plt.hist(iso_scores_all, bins=80, alpha=0.7)
    plt.axvline(percentile_val, color='r', linestyle='--', label=f'{iso_percentile}th pct = {percentile_val:.4f}')
    plt.title("Anomaly score distribution")
    plt.xlabel("Anomaly score")
    plt.ylabel("Count")
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(results_dir, "anomaly_score_histogram.png"))
    plt.close()

    # Split data
    X_train, X_test, y_train, y_test, iso_scores_train, iso_scores_test = train_test_split(
        X_scaled, y, iso_scores_all, test_size=test_size, random_state=random_state, stratify=y
    )

    print(f"[INFO] Train: {X_train.shape[0]} | Test: {X_test.shape[0]}")

    rf = RandomForestClassifier(n_estimators=150, max_depth=15, random_state=random_state, n_jobs=-1)
    gb = GradientBoostingClassifier(n_estimators=100, learning_rate=0.08, max_depth=5, random_state=random_state)
    ensemble = VotingClassifier([("rf", rf), ("gb", gb)], voting="soft")

    train_accs, val_accs, train_losses, val_losses = [], [], [], []

    for e in range(1, epochs + 1):
        ensemble.fit(X_train, y_train)
        train_preds = ensemble.predict(X_train)
        val_preds = ensemble.predict(X_test)

        train_acc = accuracy_score(y_train, train_preds)
        val_acc = accuracy_score(y_test, val_preds)

        train_accs.append(train_acc)
        val_accs.append(val_acc)
        train_losses.append(1 - train_acc)
        val_losses.append(1 - val_acc)

        print(f"[EPOCH {e}/{epochs}] Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f}")

    # Save accuracy/loss plot
    plt.figure(figsize=(10, 4))
    plt.subplot(1,2,1)
    plt.plot(range(1, len(train_accs)+1), train_accs, 'o-', label='train_acc')
    plt.plot(range(1, len(val_accs)+1), val_accs, 's-', label='val_acc')
    plt.xlabel('Epoch'); plt.ylabel('Accuracy'); plt.legend(); plt.title('Accuracy over Epochs')
    plt.subplot(1,2,2)
    plt.plot(range(1, len(train_losses)+1), train_losses, 'o-', label='train_loss')
    plt.plot(range(1, len(val_losses)+1), val_losses, 's-', label='val_loss')
    plt.xlabel('Epoch'); plt.ylabel('Loss'); plt.legend(); plt.title('Loss over Epochs')
    plt.tight_layout()
    plt.savefig(os.path.join(results_dir, "training_curves.png"))
    plt.close()

 
    # Evaluate
    ensemble_proba_test = ensemble.predict_proba(X_test)[:, 1]
    iso_flag_test = (iso_scores_test > percentile_val).astype(int)
    w_e, w_i = ensemble_weight, iso_weight
    w_e, w_i = w_e / (w_e + w_i), w_i / (w_e + w_i)
    combined_score = w_e * ensemble_proba_test + w_i * iso_flag_test
    y_pred = (combined_score >= 0.5).astype(int)

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    auc = roc_auc_score(y_test, ensemble_proba_test)

    print("\n[RESULTS]")
    print(f"Accuracy : {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall   : {rec:.4f}")
    print(f"F1 Score : {f1:.4f}")
    print(f"AUC      : {auc:.4f}")

    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(6,5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Benign','Malicious'], yticklabels=['Benign','Malicious'])
    plt.title("Confusion Matrix")
    plt.xlabel("Predicted"); plt.ylabel("Actual")
    plt.tight_layout()
    plt.savefig(os.path.join(results_dir, "confusion_matrix.png"))
    plt.close()

    # Save models
    joblib.dump(ensemble, os.path.join(saved_models_dir, "ensemble_model.pkl"))
    joblib.dump(iso, os.path.join(saved_models_dir, "isolation_forest_model.pkl"))
    joblib.dump(scaler, os.path.join(saved_models_dir, "scaler.pkl"))
    joblib.dump({
        "iso_percentile_value": float(percentile_val),
        "ensemble_weight": float(w_e),
        "iso_weight": float(w_i),
        "keep_cols_mask": keep_cols_mask.tolist(),
    }, os.path.join(saved_models_dir, "model_meta.pkl"))

    print("\n[COMPLETE] Training finished.")
    print(f"Results saved in: {results_dir}")
    print(f"Models saved in: {saved_models_dir}")
    print(f"Threshold score = {percentile_val:.6f}")

if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--benign_dir", type=str, default="dataset/benign")
    parser.add_argument("--malicious_dir", type=str, default="dataset/malicious")
    parser.add_argument("--results_dir", type=str, default="results")
    parser.add_argument("--saved_models_dir", type=str, default="saved_models")
    parser.add_argument("--iso_percentile", type=int, default=95)
    parser.add_argument("--ensemble_weight", type=float, default=0.6)
    parser.add_argument("--iso_weight", type=float, default=0.4)
    parser.add_argument("--test_size", type=float, default=0.2)
    parser.add_argument("--epochs", type=int, default=3)
    args, _ = parser.parse_known_args()

    main(
        benign_dir=args.benign_dir,
        malicious_dir=args.malicious_dir,
        results_dir=args.results_dir,
        saved_models_dir=args.saved_models_dir,
        iso_percentile=args.iso_percentile,
        ensemble_weight=args.ensemble_weight,
        iso_weight=args.iso_weight,
        test_size=args.test_size,
        epochs=args.epochs,
    )


[STEP] Loading features...
[INFO] Found 159062 images in dataset/benign


Extracting benign: 100%|██████████| 159062/159062 [1:13:16<00:00, 36.18it/s]


[INFO] Found 34388 images in dataset/malicious


Extracting malicious: 100%|██████████| 34388/34388 [16:13<00:00, 35.33it/s]


[INFO] Raw feature shape: (193450, 1781), labels: (193450,)
[INFO] After cleaning: (193450, 1781)
[INFO] IsolationForest 95th percentile = -0.011905
[INFO] Train: 154760 | Test: 38690
[EPOCH 1/3] Train Acc: 0.9999 | Val Acc: 0.9985
[EPOCH 2/3] Train Acc: 0.9999 | Val Acc: 0.9985
[EPOCH 3/3] Train Acc: 0.9999 | Val Acc: 0.9985

[RESULTS]
Accuracy : 0.9953
Precision: 0.9899
Recall   : 0.9837
F1 Score : 0.9868
AUC      : 1.0000

[COMPLETE] Training finished.
Results saved in: results
Models saved in: saved_models
Threshold score = -0.011905
